# VayuVision — Data Cleaning and Quality Scoring

## Objective

This notebook cleans the raw city-level air-quality data for Delhi, Mumbai, and Bengaluru.

The cleaning process prepares the data for time-series feature engineering and next-day AQI forecasting.

## Cleaning rules

- Convert Date into datetime format.
- Keep only Delhi, Mumbai, and Bengaluru.
- Sort records by City and Date.
- Remove duplicate rows.
- Record missing pollutant values before imputation.
- Fill pollutant null values using the median for the same city.
- Drop records where AQI is missing because AQI is the modelling target.
- Flag AQI and PM2.5 outliers using the IQR method.
- Create a data-quality score between 0 and 100.

Outliers are retained because high AQI values may represent genuine severe-pollution events.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_PATH = Path("data/raw/city_day.csv")

if not RAW_PATH.exists():
    RAW_PATH = Path("../data/raw/city_day.csv")

raw_df = pd.read_csv(RAW_PATH)

target_cities = ["Delhi", "Mumbai", "Bengaluru"]

pollutant_columns = [
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3",
    "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"
]


cleaning_df = raw_df[
    raw_df["City"].isin(target_cities)
].copy()

cleaning_df["Date"] = pd.to_datetime(
    cleaning_df["Date"],
    errors="coerce"
)

print("Raw dataset shape:", raw_df.shape)
print("Selected-city dataset shape:", cleaning_df.shape)
print("Invalid or missing dates:", cleaning_df["Date"].isna().sum())
print("Duplicate rows before cleaning:", cleaning_df.duplicated().sum())

display(
    cleaning_df[
        ["City", "Date", "AQI"] +pollutant_columns[:3]
    ].head()
)

Raw dataset shape: (29531, 16)
Selected-city dataset shape: (6027, 16)
Invalid or missing dates: 0
Duplicate rows before cleaning: 0


,City,Date,AQI,PM2.5,PM10,NO
4294,Bengaluru,2015-01-01,NaN,NaN,NaN,3.26
4295,Bengaluru,2015-01-02,NaN,NaN,NaN,6.05
4296,Bengaluru,2015-01-03,NaN,NaN,NaN,11.91
4297,Bengaluru,2015-01-04,NaN,NaN,NaN,7.45
4298,Bengaluru,2015-01-05,NaN,NaN,NaN,9.52


In [2]:
rows_before_cleaning = len(cleaning_df)

cleaning_df = (
    cleaning_df
    .dropna(subset=["Date"])
    .sort_values(["City", "Date"])
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_after_cleaning = len(cleaning_df)

# Count original missing pollutant values before filling them.
cleaning_df["missing_value_count"] = (
    cleaning_df[pollutant_columns]
    .isna()
    .sum(axis=1)
)

missing_by_city_before_imputation = (
    cleaning_df
    .groupby("City")[pollutant_columns]
    .apply(lambda city_data: city_data.isna().sum())
)

print("Rows before sorting and duplicate removal:", rows_before_cleaning)
print("Rows after sorting and duplicate removal:", rows_after_cleaning)

print("\nMissing pollutant values per city before imputation:")
display(missing_by_city_before_imputation)

print("\nDistribution of missing pollutant values per row:")
display(cleaning_df["missing_value_count"].value_counts().sort_index())

Rows before sorting and duplicate removal: 6027
Rows after sorting and duplicate removal: 6027

Missing pollutant values per city before imputation:


,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene
City,,,,,,,,,,,,
Bengaluru,146,360,6,6,4,203,11,6,144,266,93,2009
Delhi,2,77,2,2,0,9,0,110,84,0,0,781
Mumbai,1225,1246,1242,1253,499,1614,25,1220,1212,210,993,994



Distribution of missing pollutant values per row:


missing_value_count
0     1224
1     2030
2      693
3      622
4      166
5       16
6       70
7      673
8      319
9        1
10      31
11     158
12      24
Name: count, dtype: int64

In [3]:
missing_before_imputation = (
    cleaning_df[pollutant_columns]
    .isna()
    .sum()
)

for column in pollutant_columns:
    city_median = cleaning_df.groupby("City")[column].transform("median")
    
    cleaning_df[column] = cleaning_df[column].fillna(city_median)

missing_after_imputation = (
    cleaning_df[pollutant_columns]
    .isna()
    .sum()
)

imputation_summary = pd.DataFrame({
    "Missing before imputation": missing_before_imputation,
    "Missing after imputation": missing_after_imputation
})

display(imputation_summary)

,Missing before imputation,Missing after imputation
PM2.5,1373,0
PM10,1683,0
NO,1250,0
NO2,1261,0
NOx,503,0
NH3,1826,0
CO,36,0
SO2,1336,0
O3,1440,0
Benzene,476,0


In [4]:
aqi_missing_before_drop = cleaning_df["AQI"].isna().sum()

cleaned_df = (
    cleaning_df
    .dropna(subset=["AQI"])
    .copy()
)

print("Rows before dropping missing AQI:", len(cleaning_df))
print("Rows with missing AQI:", aqi_missing_before_drop)
print("Rows after dropping missing AQI:", len(cleaned_df))

display(cleaned_df[["City", "Date", "AQI", "missing_value_count"]].head())

Rows before dropping missing AQI: 6027
Rows with missing AQI: 1343
Rows after dropping missing AQI: 4684


,City,Date,AQI,missing_value_count
79,Bengaluru,2015-03-21,91.0,2
80,Bengaluru,2015-03-22,120.0,2
81,Bengaluru,2015-03-23,154.0,2
82,Bengaluru,2015-03-24,119.0,2
83,Bengaluru,2015-03-25,232.0,2


In [5]:
def get_iqr_outlier_flags(dataframe, column):
    q1 = dataframe.groupby("City")[column].transform(
        lambda values: values.quantile(0.25)
    )
    
    q3 = dataframe.groupby("City")[column].transform(
        lambda values: values.quantile(0.75)
    )
    
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    return (
        (dataframe[column] < lower_bound) |
        (dataframe[column] > upper_bound)
    )

cleaned_df["is_aqi_outlier"] = get_iqr_outlier_flags(
    cleaned_df,
    "AQI"
)

cleaned_df["is_pm25_outlier"] = get_iqr_outlier_flags(
    cleaned_df,
    "PM2.5"
)

outlier_summary = (
    cleaned_df
    .groupby("City")[["is_aqi_outlier", "is_pm25_outlier"]]
    .agg(["sum", "mean"])
)

display(outlier_summary)

is_aqi_outlier           is_pm25_outlier          
                     sum      mean             sum      mean
City                                                        
Bengaluru             71  0.037173              63  0.032984
Delhi                  6  0.003002              73  0.036518
Mumbai                 8  0.010323               5  0.006452

In [6]:
cleaned_df["data_quality_score"] = (
    100
    - (cleaned_df["missing_value_count"] * 5)
    - (cleaned_df["is_aqi_outlier"].astype(int) * 15)
).clip(0, 100).astype(int)

quality_summary = (
    cleaned_df
    .groupby("City")
    .agg(
        Records=("City", "size"),
        Average_missing_values=("missing_value_count", "mean"),
        AQI_outliers=("is_aqi_outlier", "sum"),
        Average_quality_score=("data_quality_score", "mean"),
        Minimum_quality_score=("data_quality_score", "min")
    )
    .round(2)
)

display(quality_summary)

display(
    cleaned_df[
        [
            "City", "Date", "AQI",
            "missing_value_count",
            "is_aqi_outlier",
            "data_quality_score"
        ]
    ].head(10)
)

,Records,Average_missing_values,AQI_outliers,Average_quality_score,Minimum_quality_score
City,,,,,
Bengaluru,1910,1.51,71,91.88,40
Delhi,1999,0.51,6,97.38,60
Mumbai,775,2.63,8,86.71,40


,City,Date,AQI,missing_value_count,is_aqi_outlier,data_quality_score
79,Bengaluru,2015-03-21,91.0,2,False,90
80,Bengaluru,2015-03-22,120.0,2,False,90
81,Bengaluru,2015-03-23,154.0,2,False,90
82,Bengaluru,2015-03-24,119.0,2,False,90
83,Bengaluru,2015-03-25,232.0,2,True,75
84,Bengaluru,2015-03-26,132.0,2,False,90
85,Bengaluru,2015-03-27,123.0,2,False,90
86,Bengaluru,2015-03-28,152.0,2,False,90
87,Bengaluru,2015-03-29,143.0,2,False,90
88,Bengaluru,2015-03-30,80.0,2,False,90


In [7]:
PROCESSED_PATH = Path("data/processed/cleaned_city_day.csv")

if not PROCESSED_PATH.parent.exists():
    PROCESSED_PATH = Path("../data/processed/cleaned_city_day.csv")

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

cleaned_df.to_csv(PROCESSED_PATH, index=False)

print(f"Cleaned file saved to: {PROCESSED_PATH}")
print("Final cleaned dataset shape:", cleaned_df.shape)

print("\nRemaining missing values:")
display(
    cleaned_df.isna().sum()
    .loc[lambda values: values > 0]
    .to_frame(name="Remaining missing values")
)

Cleaned file saved to: ..\data\processed\cleaned_city_day.csv
Final cleaned dataset shape: (4684, 20)

Remaining missing values:


,Remaining missing values
Xylene,1910


# Data Cleaning Conclusion

- The analysis was limited to Delhi, Mumbai, and Bengaluru.
- Dates were converted to datetime, records were sorted by city and date, and duplicate rows were checked.
- No invalid dates or duplicate rows were found in the selected-city data.
- Pollutant missing values were imputed using city-level medians where a valid median existed.
- AQI-missing records were removed because AQI is the forecasting target.
- AQI and PM2.5 outliers were flagged using city-specific IQR bounds and retained for analysis.
- Data quality scores were created using original missing-value counts and AQI-outlier flags.
- The cleaned dataset is saved for the next feature-engineering phase.